# rETH Competitiveness Study: notebook v2

*Version 2.1 adds stETH net flows as a market-wide control, Holm-adjusted p-values and a quarterly view.*

**Author:** Nishikawa. Written with AI assistance (Claude) and reviewed by the author.

This notebook runs the whole study on public data:

1. **Block numbers** for each week (DefiLlama coins API, free).
2. **On-chain series** at each block (an Ethereum archive node):
   rETH supply and exchange rate, and the stETH exchange rate as benchmark.
3. **Market prices** of rETH and ETH (DefiLlama coins API, free).
4. **Weekly panel**: net APR of rETH and stETH, yield gap, net rETH flows, market premium or discount.
5. **Self-check** against published values, then charts, a first summary and the break-even table.
6. **Demand response**: how weekly rETH flows move with the yield gap (regression with robust errors).
7. **Fee scenarios**: 12-month projection of rETH size, holder APR, node operator and RPL voter income
   for 14%, 12% and 10% fees.

**How to run:** in Google Colab, `Runtime → Run all`. The first run takes roughly 5 to 15 minutes.
Every download is cached as CSV in the `reth_study` folder, so a rerun only fetches new weeks.

**Archive node:** the notebook first tries free public endpoints. If none of them serves old blocks,
create a free Alchemy account, copy the API key, and paste it into `ALCHEMY_KEY` in the next cell.

In [ ]:
!pip -q install "web3>=6.15,<8" statsmodels

## 1. Settings

In [ ]:
import os, time, math, json
import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from web3 import Web3

START_DATE = "2024-01-06"   # first Saturday of 2024; the weekly grid runs on Saturdays, 00:00 UTC
END_DATE = None             # None = up to today
OUT_DIR = "reth_study"
os.makedirs(OUT_DIR, exist_ok=True)

# Optional: a free Alchemy key, only if the public endpoints fail. Never share this key.
# On GitHub Actions it comes from the repository secret ALCHEMY_KEY.
ALCHEMY_KEY = os.environ.get("ALCHEMY_KEY", "")

PUBLIC_RPCS = [
    "https://eth.drpc.org",
    "https://ethereum-rpc.publicnode.com",
    "https://eth.llamarpc.com",
    "https://1rpc.io/eth",
    "https://rpc.payload.de",
    "https://eth.merkle.io",
    "https://ethereum.blockpi.network/v1/rpc/public",
]

RETH = "0xae78736Cd615f374D3085123A210448E74Fc6393"   # Rocket Pool rETH
STETH = "0xae7ab96520DE3A18E5e111B5EaAb095312D7fE84"  # Lido stETH

# Published values used by the self-check (sources in the study document)
CHECKS = [
    {"label": "Rocket Pool bi-weekly update", "date": "2025-09-09", "reth_supply": 399_946},
    {"label": "Envio telemetry", "date": "2026-10-24", "reth_supply": 309_410},
]
CHECK_TOLERANCE = 0.01   # 1%

COINS = {"reth": "coingecko:rocket-pool-eth", "eth": "coingecko:ethereum"}

SATURN1 = "2026-02-18"                           # Saturn 1 mainnet launch
TRANSITION = ("2026-01-15", "2026-04-15")         # weeks dropped in the robustness check

# Fee scenarios: (node operator share, RPL voter share). Today: 5% + 9% = 14%.
FEE_NOW = (0.05, 0.09)
SCENARIOS = {
    "14% (5% + 9%), today": (0.05, 0.09),
    "12% (5% + 7%)": (0.05, 0.07),
    "10% (5% + 5%)": (0.05, 0.05),
    "10% (4% + 6%)": (0.04, 0.06),
}
LIDO_FEE = 0.10          # used to back out the gross staking APR from stETH
HORIZON_WEEKS = 52
# Share of rETH's ETH that a fee change applies to (megapools). 1.0 = full migration.
# Rerun with today's megapool share to see the near-term effect.
FEE_APPLIES_TO = 1.0

## 2. Helper functions

In [ ]:
def http_json(method, url, retries=5, backoff=2.0, **kwargs):
    # GET or POST with retries; returns parsed JSON
    last = None
    for i in range(retries):
        try:
            r = requests.request(method, url, timeout=30, **kwargs)
            if r.status_code == 429:
                raise RuntimeError("rate limited (HTTP 429)")
            r.raise_for_status()
            return r.json()
        except Exception as e:
            last = e
            time.sleep(backoff * (i + 1))
    raise RuntimeError(f"{method} {url} failed after {retries} tries: {last}")


def weekly_dates(start, end=None):
    start = pd.Timestamp(start, tz="UTC")
    end = pd.Timestamp.now(tz="UTC").normalize() if end is None else pd.Timestamp(end, tz="UTC")
    return list(pd.date_range(start, end, freq="7D"))


def to_unix(ts):
    return int(pd.Timestamp(ts).timestamp())


def block_at_llama(unix_ts):
    data = http_json("GET", f"https://coins.llama.fi/block/ethereum/{unix_ts}")
    height = data.get("height", data.get("block"))
    if height is None:
        raise RuntimeError(f"Unexpected block response: {data}")
    return int(height)


def block_at_rpc(w3, unix_ts, max_steps=12):
    # Fallback: secant search on block timestamps (about 12 s per block)
    latest = w3.eth.get_block("latest")
    n = int(latest["number"] - (latest["timestamp"] - unix_ts) / 12)
    for _ in range(max_steps):
        b = w3.eth.get_block(n)
        diff = unix_ts - b["timestamp"]
        if abs(diff) < 24:
            return n
        n += int(diff / 12)
    return n


def load_csv(name):
    path = os.path.join(OUT_DIR, name)
    if os.path.exists(path):
        df = pd.read_csv(path)
        if "date" in df.columns:
            df["date"] = pd.to_datetime(df["date"], utc=True)
        return df
    return None


def save_csv(df, name):
    df.to_csv(os.path.join(OUT_DIR, name), index=False)


RETH_ABI = [
    {"inputs": [], "name": "totalSupply", "outputs": [{"name": "", "type": "uint256"}],
     "stateMutability": "view", "type": "function"},
    {"inputs": [], "name": "getExchangeRate", "outputs": [{"name": "", "type": "uint256"}],
     "stateMutability": "view", "type": "function"},
]
STETH_ABI = [
    {"inputs": [{"name": "_sharesAmount", "type": "uint256"}], "name": "getPooledEthByShares",
     "outputs": [{"name": "", "type": "uint256"}], "stateMutability": "view", "type": "function"},
    {"inputs": [], "name": "getTotalPooledEther", "outputs": [{"name": "", "type": "uint256"}],
     "stateMutability": "view", "type": "function"},
]


def connect_archive(test_block=None):
    # Returns a Web3 connection that can read state at the oldest block we need
    candidates = []
    if ALCHEMY_KEY:
        candidates.append(("Alchemy", f"https://eth-mainnet.g.alchemy.com/v2/{ALCHEMY_KEY}"))
    candidates += [(url, url) for url in PUBLIC_RPCS]
    for label, url in candidates:
        try:
            w3 = Web3(Web3.HTTPProvider(url, request_kwargs={"timeout": 30}))
            latest = w3.eth.block_number
            block = test_block if test_block else latest - 2_000_000
            reth = w3.eth.contract(address=Web3.to_checksum_address(RETH), abi=RETH_ABI)
            old_supply = reth.functions.totalSupply().call(block_identifier=block)
            if old_supply > 0:
                print(f"Archive RPC OK: {label} (tested at block {block})")
                return w3
        except Exception as e:
            print(f"Skipping {label}: {type(e).__name__}: {str(e)[:100]}")
    raise RuntimeError("No archive endpoint worked. Paste a free Alchemy key into ALCHEMY_KEY and rerun.")


def merge_rows(existing, rows):
    new = pd.DataFrame(rows)
    return new if existing is None else pd.concat([existing, new], ignore_index=True)


def read_state(w3, block):
    reth = w3.eth.contract(address=Web3.to_checksum_address(RETH), abi=RETH_ABI)
    steth = w3.eth.contract(address=Web3.to_checksum_address(STETH), abi=STETH_ABI)
    for attempt in range(4):
        try:
            return {
                "reth_supply": reth.functions.totalSupply().call(block_identifier=block) / 1e18,
                "reth_rate": reth.functions.getExchangeRate().call(block_identifier=block) / 1e18,
                "steth_rate": steth.functions.getPooledEthByShares(10**18).call(block_identifier=block) / 1e18,
                "steth_pooled_eth": steth.functions.getTotalPooledEther().call(block_identifier=block) / 1e18,
            }
        except Exception:
            if attempt == 3:
                raise
            time.sleep(2 * (attempt + 1))


def read_steth_pooled(w3, block):
    # Used to backfill weeks cached before this column existed
    steth = w3.eth.contract(address=Web3.to_checksum_address(STETH), abi=STETH_ABI)
    for attempt in range(4):
        try:
            return steth.functions.getTotalPooledEther().call(block_identifier=block) / 1e18
        except Exception:
            if attempt == 3:
                raise
            time.sleep(2 * (attempt + 1))


def price_at(unix_ts):
    ids = ",".join(COINS.values())
    data = http_json("GET", f"https://coins.llama.fi/prices/historical/{unix_ts}/{ids}",
                     params={"searchWidth": "6h"})
    coins = data.get("coins", {})
    out = {}
    for key, cid in COINS.items():
        out[f"{key}_usd"] = coins.get(cid, {}).get("price", np.nan)
    return out

## 3. Block number for each week

In [ ]:
dates = weekly_dates(START_DATE, END_DATE)
blocks = load_csv("blocks.csv")
known = set() if blocks is None else set(blocks["date"])
new_rows = []
for d in dates:
    if d in known:
        continue
    unix_ts = to_unix(d)
    try:
        b = block_at_llama(unix_ts)
    except Exception as e:
        print(f"Block lookup failed for {d.date()} ({e}); will use the RPC fallback later")
        b = None
    new_rows.append({"date": d, "unix": unix_ts, "block": b})
    time.sleep(0.1)
if new_rows:
    blocks = merge_rows(blocks, new_rows)
    save_csv(blocks, "blocks.csv")
print(f"{len(blocks)} weeks, from {blocks['date'].min().date()} to {blocks['date'].max().date()}")

## 4. On-chain series at each block (archive node)

In [ ]:
known_blocks = blocks["block"].dropna()
w3 = connect_archive(int(known_blocks.min()) if len(known_blocks) else None)

missing = blocks["block"].isna()
for i in blocks.index[missing]:
    blocks.loc[i, "block"] = block_at_rpc(w3, int(blocks.loc[i, "unix"]))
blocks["block"] = blocks["block"].astype(int)
save_csv(blocks, "blocks.csv")

onchain = load_csv("onchain.csv")
if onchain is not None and "steth_pooled_eth" not in onchain.columns:
    onchain["steth_pooled_eth"] = np.nan
if onchain is not None and onchain["steth_pooled_eth"].isna().any():
    todo = onchain.index[onchain["steth_pooled_eth"].isna()]
    print(f"Backfilling stETH pooled ETH for {len(todo)} cached weeks...")
    for n, i in enumerate(todo, 1):
        onchain.loc[i, "steth_pooled_eth"] = read_steth_pooled(w3, int(onchain.loc[i, "block"]))
        if n % 20 == 0:
            save_csv(onchain, "onchain.csv")
        time.sleep(0.2)
    save_csv(onchain, "onchain.csv")
done = set() if onchain is None else set(onchain["date"])
rows = []
for _, row in blocks.iterrows():
    if row["date"] in done:
        continue
    state = read_state(w3, int(row["block"]))
    rows.append({"date": row["date"], "block": int(row["block"]), **state})
    if len(rows) % 20 == 0:
        save_csv(merge_rows(onchain, rows), "onchain.csv")   # keep progress if the run stops
        print(f"  read {len(rows)} new weeks...")
    time.sleep(0.2)
if rows:
    onchain = merge_rows(onchain, rows)
    save_csv(onchain, "onchain.csv")
print(f"On-chain rows: {len(onchain)}")

## 5. Market prices of rETH and ETH

In [ ]:
prices = load_csv("prices.csv")
done = set() if prices is None else set(prices["date"])
rows = []
for _, row in blocks.iterrows():
    if row["date"] in done:
        continue
    rows.append({"date": row["date"], **price_at(int(row["unix"]))})
    if len(rows) % 20 == 0:
        save_csv(merge_rows(prices, rows), "prices.csv")   # keep progress if the run stops
    time.sleep(0.1)
if rows:
    prices = merge_rows(prices, rows)
    save_csv(prices, "prices.csv")
print(f"Price rows: {len(prices)}; missing rETH prices: {int(prices['reth_usd'].isna().sum())}")

## 6. Weekly panel and self-check

In [ ]:
def build_panel(onchain, prices):
    df = onchain.merge(prices, on="date", how="left").sort_values("date").reset_index(drop=True)
    days = df["date"].diff().dt.total_seconds() / 86400
    df["reth_backing_eth"] = df["reth_supply"] * df["reth_rate"]
    df["reth_apr"] = (df["reth_rate"] / df["reth_rate"].shift(1) - 1) * 365 / days
    df["steth_apr"] = (df["steth_rate"] / df["steth_rate"].shift(1) - 1) * 365 / days
    df["gap_bps"] = (df["reth_apr"] - df["steth_apr"]) * 1e4
    df["gap_bps_4w"] = df["gap_bps"].rolling(4).mean()
    df["net_flow_eth"] = df["reth_supply"].diff() * df["reth_rate"]
    df["net_flow_pct"] = df["reth_supply"].diff() / df["reth_supply"].shift(1) * 100
    df["market_ratio"] = df["reth_usd"] / df["eth_usd"]
    df["premium_pct"] = (df["market_ratio"] / df["reth_rate"] - 1) * 100
    # stETH net flow: change in pooled ETH minus the part that is rewards
    df["steth_flow_pct"] = (df["steth_pooled_eth"] / df["steth_pooled_eth"].shift(1)
                            - df["steth_rate"] / df["steth_rate"].shift(1)) * 100
    return df


def self_check(w3):
    ok = True
    for c in CHECKS:
        d = pd.Timestamp(c["date"], tz="UTC")
        unix_ts = min(to_unix(d), int(time.time()) - 600)
        try:
            b = block_at_llama(unix_ts)
        except Exception:
            b = block_at_rpc(w3, unix_ts)
        got = read_state(w3, b)["reth_supply"]
        err = got / c["reth_supply"] - 1
        flag = "OK" if abs(err) <= CHECK_TOLERANCE else "MISMATCH"
        ok = ok and flag == "OK"
        print(f"{flag}: {c['label']} {c['date']}: published {c['reth_supply']:,.0f}, on-chain {got:,.0f} ({err:+.2%})")
    if not ok:
        raise RuntimeError("Self-check failed: stop and review before using the data.")


panel = build_panel(onchain, prices)
save_csv(panel, "weekly_panel.csv")
self_check(w3)
panel.tail()

## 7. Charts

In [ ]:
saturn1 = pd.Timestamp("2026-02-18", tz="UTC")

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(panel["date"], panel["reth_supply"] / 1e3, label="rETH supply (thousand)")
ax.plot(panel["date"], panel["reth_backing_eth"] / 1e3, label="ETH behind rETH (thousand)")
ax.axvline(saturn1, color="grey", linestyle="--", linewidth=1)
ax.text(saturn1, ax.get_ylim()[1], " Saturn 1", va="top", fontsize=9, color="grey")
ax.set_title("rETH supply and the ETH behind it, weekly")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "chart_supply.png"), dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(panel["date"], panel["reth_apr"].rolling(4).mean() * 100, label="rETH net APR, 4-week average (%)")
ax.plot(panel["date"], panel["steth_apr"].rolling(4).mean() * 100, label="stETH net APR, 4-week average (%)")
ax.axvline(saturn1, color="grey", linestyle="--", linewidth=1)
ax.set_title("Net APR of rETH and stETH, from their exchange rates")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "chart_apr.png"), dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(6, 5))
x = panel["gap_bps_4w"].shift(1)
y = panel["net_flow_pct"]
ax.scatter(x, y, s=14, alpha=0.7)
ax.axhline(0, color="grey", linewidth=0.8); ax.axvline(0, color="grey", linewidth=0.8)
ax.set_xlabel("Yield gap rETH minus stETH, 4-week average, previous week (bps)")
ax.set_ylabel("Net rETH flow this week (% of supply)")
ax.set_title("Weekly rETH flows against the yield gap")
ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "chart_flows_vs_gap.png"), dpi=150); plt.show()

fig, ax = plt.subplots(figsize=(10, 3.5))
ax.plot(panel["date"], panel["premium_pct"], label="rETH market price vs protocol rate (%)")
ax.axhline(0, color="grey", linewidth=0.8)
ax.set_title("rETH premium (+) or discount (-) to its protocol rate")
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "chart_premium.png"), dpi=150); plt.show()

## 8. First summary (descriptive only)

In [ ]:
def summarise(panel, since=None):
    p = panel if since is None else panel[panel["date"] >= pd.Timestamp(since, tz="UTC")]
    lagged = pd.DataFrame({"flow": p["net_flow_pct"], "gap_prev": p["gap_bps_4w"].shift(1),
                           "prem": p["premium_pct"]}).dropna()
    return {
        "weeks": int(len(p)),
        "mean_gap_bps": round(float(p["gap_bps"].mean()), 1),
        "median_gap_bps": round(float(p["gap_bps"].median()), 1),
        "share_of_weeks_rETH_behind": round(float((p["gap_bps"] < 0).mean()), 2),
        "net_flow_eth_total": round(float(p["net_flow_eth"].sum()), 0),
        "corr_flow_vs_prev_gap": round(float(lagged["flow"].corr(lagged["gap_prev"])), 2) if len(lagged) > 5 else None,
        "corr_flow_vs_premium": round(float(lagged["flow"].corr(lagged["prem"])), 2) if len(lagged) > 5 else None,
        "median_weekly_flow_rETH_pct": round(float(p["net_flow_pct"].median()), 3),
        "median_weekly_flow_stETH_pct": round(float(p["steth_flow_pct"].median()), 3),
        "corr_rETH_vs_stETH_flows": round(float(p["net_flow_pct"].corr(p["steth_flow_pct"])), 2),
    }

summary = pd.DataFrame({
    "Since Jan 2024": summarise(panel),
    "Since Saturn 1 (18 Feb 2026)": summarise(panel, "2026-02-18"),
})
summary

In [ ]:
# Quarterly view: actual yield gap against the part the fee difference explains
q = (panel.assign(quarter=panel["date"].dt.tz_convert(None).dt.to_period("Q"))
          .groupby("quarter")
          .agg(gap_bps=("gap_bps", "median"), steth_apr=("steth_apr", "median"),
               reth_flow_pct=("net_flow_pct", "median"), steth_flow_pct=("steth_flow_pct", "median")))
q["fee_only_gap_bps"] = -(sum(FEE_NOW) - LIDO_FEE) * q["steth_apr"] / (1 - LIDO_FEE) * 1e4
save_csv(q.reset_index().astype({"quarter": str}), "quarterly.csv")

fig, ax = plt.subplots(figsize=(10, 4))
x = np.arange(len(q))
ax.bar(x - 0.2, q["gap_bps"], width=0.4, label="Actual gap, weekly median (bps)", color="#4c72b0")
ax.bar(x + 0.2, q["fee_only_gap_bps"], width=0.4, label="Part explained by the 14% vs 10% fee (bps)", color="#b0b0b0")
ax.set_xticks(x); ax.set_xticklabels([str(p) for p in q.index], rotation=45)
ax.axhline(0, color="black", linewidth=0.8); ax.grid(alpha=0.3, axis="y"); ax.legend()
ax.set_title("rETH minus stETH yield by quarter")
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "chart_quarterly_gap.png"), dpi=150); plt.show()
q.round(3)

## 9. Break-even growth for fee cuts (pure arithmetic)

In [ ]:
rows = []
for old, new in [(9, 8), (9, 7), (9, 6), (9, 5)]:
    rows.append({"change": f"Voter share {old}% -> {new}%", "fee paid by rETH holders": f"{5 + old}% -> {5 + new}%",
                 "growth needed (RPL voters)": f"+{(old / new - 1) * 100:.1f}%"})
rows.append({"change": "Node operator share 5% -> 4%", "fee paid by rETH holders": "14% -> 13%",
             "growth needed (RPL voters)": "+25.0% (node operators)"})
pd.DataFrame(rows)

## 10. Demand response: do flows follow the yield gap?

Weekly net flow (% of supply) regressed on the 4-week yield gap one to four weeks earlier,
last week's market premium, the weekly ETH price change and a Saturn 1 marker.
Standard errors are robust to autocorrelation (Newey-West, 4 lags).
`beta_gap` is the change in weekly flow, in percentage points, per basis point of yield gap.

In [ ]:
import statsmodels.api as sm

def regress(panel, lag=1, exclude=None, market=False):
    d = panel.copy()
    d["gap_lag"] = d["gap_bps_4w"].shift(lag)
    d["prem_lag"] = d["premium_pct"].shift(1)
    d["eth_ret"] = d["eth_usd"].pct_change() * 100
    d["saturn1"] = (d["date"] >= pd.Timestamp(SATURN1, tz="UTC")).astype(int)
    if exclude is not None:
        lo, hi = (pd.Timestamp(x, tz="UTC") for x in exclude)
        d = d[~d["date"].between(lo, hi)]
    cols = ["gap_lag", "prem_lag", "eth_ret", "saturn1"]
    if market:
        d["steth_flow"] = d["steth_flow_pct"]
        cols.append("steth_flow")
    d = d.dropna(subset=["net_flow_pct"] + cols)
    if d["saturn1"].nunique() < 2:
        cols.remove("saturn1")
    X = sm.add_constant(d[cols])
    return sm.OLS(d["net_flow_pct"], X).fit(cov_type="HAC", cov_kwds={"maxlags": 4})


def row_of(label, m):
    lo, hi = m.conf_int().loc["gap_lag"]
    return {"model": label, "beta_gap": m.params["gap_lag"], "ci95_low": lo, "ci95_high": hi,
            "p_value": m.pvalues["gap_lag"], "beta_premium": m.params["prem_lag"],
            "p_premium": m.pvalues["prem_lag"],
            "beta_market": m.params.get("steth_flow", np.nan), "p_market": m.pvalues.get("steth_flow", np.nan),
            "weeks": int(m.nobs), "r2": m.rsquared}

models = {f"lag {k} week(s)": regress(panel, lag=k) for k in (1, 2, 3, 4)}
models["lag 1, without Saturn 1 transition"] = regress(panel, lag=1, exclude=TRANSITION)
models["lag 1, with stETH flows as market control"] = regress(panel, lag=1, market=True)
models["lag 4, with stETH flows as market control"] = regress(panel, lag=4, market=True)
response = pd.DataFrame([row_of(k, m) for k, m in models.items()])
# Four lags were tested, so adjust their p-values for multiple testing (Holm)
from statsmodels.stats.multitest import multipletests
base = response["model"].str.fullmatch(r"lag \d week\(s\)")
response["p_holm"] = np.nan
response.loc[base, "p_holm"] = multipletests(response.loc[base, "p_value"], method="holm")[1]
save_csv(response, "demand_response.csv")
print(models["lag 1 week(s)"].summary().tables[1])
response.round(4)

## 11. Fee scenarios: 12 months ahead

Inputs come from the data: the gross staking APR (stETH APR before Lido's 10% fee, last 26 weeks),
the ETH behind rETH today, the recent flow trend (last 12 weeks) and `beta_gap` with its 95% interval.
A fee cut raises the holder APR by the cut times the gross APR, which shifts the yield gap,
which shifts weekly flows by `beta_gap` per basis point. Everything else is held constant.

In [ ]:
recent = panel.tail(26)
gross_apr = float((recent["steth_apr"] / (1 - LIDO_FEE)).median())
effective_reth_fee = float(1 - recent["reth_apr"].median() / gross_apr)
backing0 = float(panel["reth_backing_eth"].iloc[-1])
trend_flow = float(panel["net_flow_pct"].tail(12).median())
base = response.iloc[0]
betas = {"low": base["ci95_low"], "mid": base["beta_gap"], "high": base["ci95_high"]}
print(f"Gross staking APR (from stETH): {gross_apr:.3%}")
print(f"Effective rETH fee implied by the data: {effective_reth_fee:.1%} (nominal {sum(FEE_NOW):.0%})")
print(f"ETH behind rETH today: {backing0:,.0f}; recent weekly flow: {trend_flow:+.2f}%")
print(f"beta_gap (lag 1): {betas['mid']:.4f} [{betas['low']:.4f}, {betas['high']:.4f}] % per week per bps")


def project(no_share, voter_share, beta, applies=FEE_APPLIES_TO):
    fee_new = applies * (no_share + voter_share) + (1 - applies) * sum(FEE_NOW)
    apr_gain_bps = (sum(FEE_NOW) - fee_new) * gross_apr * 1e4
    weekly = trend_flow + beta * apr_gain_bps
    backing = backing0 * (1 + weekly / 100) ** HORIZON_WEEKS
    voter = applies * voter_share + (1 - applies) * FEE_NOW[1]
    operator = applies * no_share + (1 - applies) * FEE_NOW[0]
    return {"holder_apr_gain_bps": apr_gain_bps, "weekly_flow_pct": weekly, "eth_behind_reth_12m": backing,
            "holder_net_apr": gross_apr * (1 - fee_new),
            "voter_income_eth_per_year": backing * gross_apr * voter,
            "operator_income_eth_per_year": backing * gross_apr * operator}


rows = []
for name, (no_s, v_s) in SCENARIOS.items():
    for case, beta in betas.items():
        rows.append({"scenario": name, "beta_case": case, **project(no_s, v_s, beta)})
scen = pd.DataFrame(rows)
today = scen[(scen["scenario"] == list(SCENARIOS)[0]) & (scen["beta_case"] == "mid")].iloc[0]
scen["voter_income_vs_today"] = scen["voter_income_eth_per_year"] / today["voter_income_eth_per_year"] - 1
scen["operator_income_vs_today"] = scen["operator_income_eth_per_year"] / today["operator_income_eth_per_year"] - 1
save_csv(scen, "fee_scenarios.csv")

view = scen.pivot(index="scenario", columns="beta_case", values="voter_income_vs_today")[["low", "mid", "high"]]
print("RPL voter income after 12 months, change vs keeping 14% (low / mid / high response):")
(view * 100).round(1)

In [ ]:
mid = scen[scen["beta_case"] == "mid"].set_index("scenario")
err_lo = mid["voter_income_vs_today"] - scen[scen["beta_case"] == "low"].set_index("scenario")["voter_income_vs_today"]
err_hi = scen[scen["beta_case"] == "high"].set_index("scenario")["voter_income_vs_today"] - mid["voter_income_vs_today"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(mid.index, mid["voter_income_vs_today"] * 100,
       yerr=[err_lo.abs() * 100, err_hi.abs() * 100], capsize=4, color="#7f7f7f")
ax.axhline(0, color="black", linewidth=0.8)
ax.set_ylabel("Change vs keeping 14% (%)")
ax.set_title("RPL voter income after 12 months, by fee (bars: 95% range of the demand response)")
plt.xticks(rotation=15)
fig.tight_layout(); fig.savefig(os.path.join(OUT_DIR, "chart_scenarios.png"), dpi=150); plt.show()

## 12. Save and download

In [ ]:
print("Files in", OUT_DIR, ":", sorted(os.listdir(OUT_DIR)))
try:
    from google.colab import files  # only works inside Colab
    import shutil
    shutil.make_archive("reth_study_v2", "zip", OUT_DIR)
    files.download("reth_study_v2.zip")
except Exception as e:
    print("Not in Colab, or download blocked:", type(e).__name__)

## Next (v3)

- Rerun the scenarios with today's megapool share (`FEE_APPLIES_TO`).
- Small interactive app (Streamlit) and the public GitHub repository.
- Forum post with the findings.